# ECE 661 Homework 4: Lab 2 and Lab 3

Keep these files in the same folder as this notebook: `FP_layers.py`, `resnet20.py`, `train_util.py` and `pretrained_model.pt`.
In Google Colab, upload them to the session (or to a Drive folder you mount and `cd` into) before running the first code block.
CIFAR-10 is downloaded to `./data` automatically. A GPU runtime is recommended.

- Lab 2: fill in the functions and the `...` blanks in this notebook.
- Lab 3 (a): fill in the `STE` forward function in `FP_layers.py`, then restart the runtime so the change is imported.


In [1]:
import urllib.request

url = "https://raw.githubusercontent.com/codeteme/ece661_ce_ml_nn/main/hw_4/FP_layers.py"
urllib.request.urlretrieve(url, "FP_layers.py")
print("Successfully downloaded FP_layers.py!")

Successfully downloaded FP_layers.py!


In [2]:
import urllib.request

files = ["resnet20.py", "train_util.py", "pretrained_model.pt"]
base_url = "https://raw.githubusercontent.com/codeteme/ece661_ce_ml_nn/main/hw_4/"

for file in files:
    urllib.request.urlretrieve(base_url + file, file)
    print(f"Successfully downloaded {file}!")

Successfully downloaded resnet20.py!
Successfully downloaded train_util.py!
Successfully downloaded pretrained_model.pt!


### Lab2 (a) Model preparation

In [3]:
from resnet20 import ResNetCIFAR
from train_util import train, finetune, test, get_loaders
import torch
import numpy as np

import time

import torchvision.transforms as transforms
import torchvision
import torch.nn as nn
import torch.optim as optim

from FP_layers import *

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
net = ResNetCIFAR(num_layers=20, Nbits=None)
net = net.to(device)

In [5]:
print(f"{'Layer Name':<25} | {'Layer Type & Dimensions'}")
print("-" * 80)

# Inspect the first few layers that match conv or linear layers
count = 0
for name, layer in net.named_modules():
    if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
        # Format and display the layer details
        print(f"{name:<25} | {layer}")
        count += 1
        if count >= 10:
            print("... (truncated) ...")
            break

Layer Name                | Layer Type & Dimensions
--------------------------------------------------------------------------------
head_conv.0.conv          | Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
body_op.0.conv1.0.conv    | Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
body_op.0.conv2.0.conv    | Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
body_op.1.conv1.0.conv    | Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
body_op.1.conv2.0.conv    | Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
body_op.2.conv1.0.conv    | Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
body_op.2.conv2.0.conv    | Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
body_op.3.conv1.0.conv    | Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
body_op.3.conv2.0.co

In [6]:
# Load the best weight paramters
net.load_state_dict(torch.load("pretrained_model.pt", map_location=device))
test(net)

100%|██████████| 170M/170M [03:01<00:00, 938kB/s] 


Test Loss=0.3231, Test accuracy=0.9151


### Lab2 (b) Prune by percentage

In [7]:
def prune_by_percentage(layer, q=70.0):
    """
    Pruning the weight paramters by threshold.
    :param q: pruning percentile. 'q' percent of the least
    significant weight paramters will be pruned.
    """
    # Convert the weight of "layer" to numpy array
    weights = layer.weight.data.cpu().numpy()

    # Compute the q-th percentile of the abs of the converted array
    abs_weights = np.abs(weights)
    threshold = np.percentile(abs_weights, q)

    # Generate a binary mask same shape as weight to decide which element to prune
    mask = (abs_weights > threshold).astype(np.float32)

    # Convert mask to torch tensor and put on GPU
    mask_tensor = torch.from_numpy(mask).to(layer.weight.device)

    # Multiply the weight by mask to perform pruning
    layer.weight.data.mul_(mask_tensor)

In [8]:
net.load_state_dict(torch.load("pretrained_model.pt", map_location=device))

for name,layer in net.named_modules():
    if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
        # change q value (in percent, e.g. q=10.0 prunes 10%)
        prune_by_percentage(layer, q=70.0)

        # Optional: Check the sparsity you achieve in each layer
        # Convert the weight of "layer" to numpy array
        np_weight = layer.weight.data.cpu().numpy()
        # Count number of zeros
        zeros = np_weight.size - np.count_nonzero(np_weight)
        # Count number of parameters
        total = np_weight.size
        # Print sparsity
        print('Sparsity of '+name+': '+str(zeros/total))

test(net)

Sparsity of head_conv.0.conv: 0.6990740740740741
Sparsity of body_op.0.conv1.0.conv: 0.7000868055555556
Sparsity of body_op.0.conv2.0.conv: 0.7000868055555556
Sparsity of body_op.1.conv1.0.conv: 0.7000868055555556
Sparsity of body_op.1.conv2.0.conv: 0.7000868055555556
Sparsity of body_op.2.conv1.0.conv: 0.7000868055555556
Sparsity of body_op.2.conv2.0.conv: 0.7000868055555556
Sparsity of body_op.3.conv1.0.conv: 0.6998697916666666
Sparsity of body_op.3.conv2.0.conv: 0.6999782986111112
Sparsity of body_op.4.conv1.0.conv: 0.6999782986111112
Sparsity of body_op.4.conv2.0.conv: 0.6999782986111112
Sparsity of body_op.5.conv1.0.conv: 0.6999782986111112
Sparsity of body_op.5.conv2.0.conv: 0.6999782986111112
Sparsity of body_op.6.conv1.0.conv: 0.6999782986111112
Sparsity of body_op.6.conv2.0.conv: 0.7000054253472222
Sparsity of body_op.7.conv1.0.conv: 0.7000054253472222
Sparsity of body_op.7.conv2.0.conv: 0.7000054253472222
Sparsity of body_op.8.conv1.0.conv: 0.7000054253472222
Sparsity of body

### Lab2 (c) Finetune pruned model

In [11]:
def finetune_after_prune(net, trainloader, criterion, optimizer, prune=True):
    """
    Finetune the pruned model for a single epoch
    Make sure pruned weights are kept as zero
    """
    # Build a dictionary for the nonzero weights
    weight_mask = {}
    for name,layer in net.named_modules():
        if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
            # Your code here: generate a mask in GPU torch tensor to have 1 for nonzero element and 0 for zero element
            weight_mask[name] = (layer.weight != 0).float()

    global_steps = 0
    train_loss = 0
    correct = 0
    total = 0
    start = time.time()
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        if prune:
            for name,layer in net.named_modules():
                if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
                    # Your code here: Use weight_mask to make sure zero elements remains zero
                    layer.weight.data.mul_(weight_mask[name])

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        global_steps += 1

        if global_steps % 50 == 0:
            end = time.time()
            batch_size = 256
            num_examples_per_second = 50 * batch_size / (end - start)
            print("[Step=%d]\tLoss=%.4f\tacc=%.4f\t%.1f examples/second"
                 % (global_steps, train_loss / (batch_idx + 1), (correct / total), num_examples_per_second))
            start = time.time()

In [12]:
# Get pruned model
net.load_state_dict(torch.load("pretrained_model.pt", map_location=device))
for name,layer in net.named_modules():
    if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
        prune_by_percentage(layer, q=85) # adjust this

# Training setup, do not change
batch_size=256
lr=0.002
reg=1e-4

print('==> Preparing data..')
# CIFAR-10 is kept in device memory and augmented there (same random crop and flip as before), see train_util.py
trainloader, testloader = get_loaders(batch_size=batch_size, device=device)
best_acc = 0  # best test accuracy
start_epoch = 0  # start from epoch 0 or last checkpoint epoch

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=lr, momentum=0.875, weight_decay=reg, nesterov=False)

==> Preparing data..


In [13]:
# Model finetuning
for epoch in range(20):
    print('\nEpoch: %d' % epoch)
    net.train()
    finetune_after_prune(net, trainloader, criterion, optimizer)
    #Start the testing code.
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    num_val_steps = len(testloader)
    val_acc = correct / total
    print("Test Loss=%.4f, Test acc=%.4f" % (test_loss / (num_val_steps), val_acc))

    if val_acc > best_acc:
        best_acc = val_acc
        print("Saving...")
        torch.save(net.state_dict(), "net_after_finetune.pt")


Epoch: 0
[Step=50]	Loss=1.4884	acc=0.5276	4222.4 examples/second
[Step=100]	Loss=1.2289	acc=0.6020	5694.6 examples/second
[Step=150]	Loss=1.0910	acc=0.6436	5684.9 examples/second
Test Loss=0.7482, Test acc=0.7579
Saving...

Epoch: 1
[Step=50]	Loss=0.7081	acc=0.7622	5624.4 examples/second
[Step=100]	Loss=0.6825	acc=0.7701	5642.4 examples/second
[Step=150]	Loss=0.6632	acc=0.7773	5628.6 examples/second
Test Loss=0.6403, Test acc=0.7933
Saving...

Epoch: 2
[Step=50]	Loss=0.5923	acc=0.7994	5590.4 examples/second
[Step=100]	Loss=0.5807	acc=0.8040	5548.9 examples/second
[Step=150]	Loss=0.5763	acc=0.8051	5559.4 examples/second
Test Loss=0.5910, Test acc=0.8070
Saving...

Epoch: 3
[Step=50]	Loss=0.5413	acc=0.8179	5531.3 examples/second
[Step=100]	Loss=0.5291	acc=0.8221	5515.2 examples/second
[Step=150]	Loss=0.5253	acc=0.8239	5467.2 examples/second
Test Loss=0.5618, Test acc=0.8146
Saving...

Epoch: 4
[Step=50]	Loss=0.5012	acc=0.8321	5434.9 examples/second
[Step=100]	Loss=0.4939	acc=0.8368	5424

In [14]:
# Check sparsity of the finetuned model, make sure it's not changed
net.load_state_dict(torch.load("net_after_finetune.pt", map_location=device))

for name,layer in net.named_modules():
    if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
        # Your code here:
        # Convert the weight of "layer" to numpy array
        np_weight = layer.weight.data.cpu().numpy()
        # Count number of zeros
        zeros = np_weight.size - np.count_nonzero(np_weight)
        # Count number of parameters
        total = np_weight.size
        # Print sparsity
        print('Sparsity of '+name+': '+str(zeros/total))

test(net)

Sparsity of head_conv.0.conv: 0.8495370370370371
Sparsity of body_op.0.conv1.0.conv: 0.8498263888888888
Sparsity of body_op.0.conv2.0.conv: 0.8498263888888888
Sparsity of body_op.1.conv1.0.conv: 0.8498263888888888
Sparsity of body_op.1.conv2.0.conv: 0.8498263888888888
Sparsity of body_op.2.conv1.0.conv: 0.8498263888888888
Sparsity of body_op.2.conv2.0.conv: 0.8498263888888888
Sparsity of body_op.3.conv1.0.conv: 0.8498263888888888
Sparsity of body_op.3.conv2.0.conv: 0.8499348958333334
Sparsity of body_op.4.conv1.0.conv: 0.8499348958333334
Sparsity of body_op.4.conv2.0.conv: 0.8499348958333334
Sparsity of body_op.5.conv1.0.conv: 0.8499348958333334
Sparsity of body_op.5.conv2.0.conv: 0.8499348958333334
Sparsity of body_op.6.conv1.0.conv: 0.8499891493055556
Sparsity of body_op.6.conv2.0.conv: 0.8499891493055556
Sparsity of body_op.7.conv1.0.conv: 0.8499891493055556
Sparsity of body_op.7.conv2.0.conv: 0.8499891493055556
Sparsity of body_op.8.conv1.0.conv: 0.8499891493055556
Sparsity of body

### Lab2 (d) Iterative pruning

In [15]:
net.load_state_dict(torch.load("pretrained_model.pt", map_location=device))
best_acc = 0.
for epoch in range(20):
    print('\nEpoch: %d' % epoch)

    net.train()
    if epoch<10:
        for name,layer in net.named_modules():
            if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
                # Increase model sparsity
                q = 8.5 * (epoch + 1)
                prune_by_percentage(layer, q=q)
    if epoch<9:
        finetune_after_prune(net, trainloader, criterion, optimizer,prune=False)
    else:
        finetune_after_prune(net, trainloader, criterion, optimizer)

    #Start the testing code.
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    num_val_steps = len(testloader)
    val_acc = correct / total
    print("Test Loss=%.4f, Test acc=%.4f" % (test_loss / (num_val_steps), val_acc))

    if epoch>=10:
        if val_acc > best_acc:
            best_acc = val_acc
            print("Saving...")
            torch.save(net.state_dict(), "net_after_iterative_prune.pt")


Epoch: 0
[Step=50]	Loss=0.0457	acc=0.9855	5377.6 examples/second
[Step=100]	Loss=0.0470	acc=0.9850	5549.0 examples/second
[Step=150]	Loss=0.0473	acc=0.9853	5533.8 examples/second
Test Loss=0.3256, Test acc=0.9146

Epoch: 1
[Step=50]	Loss=0.0490	acc=0.9843	5505.4 examples/second
[Step=100]	Loss=0.0469	acc=0.9853	5476.4 examples/second
[Step=150]	Loss=0.0481	acc=0.9850	5432.0 examples/second
Test Loss=0.3289, Test acc=0.9134

Epoch: 2
[Step=50]	Loss=0.0548	acc=0.9824	5404.1 examples/second
[Step=100]	Loss=0.0511	acc=0.9834	5377.9 examples/second
[Step=150]	Loss=0.0510	acc=0.9834	5376.6 examples/second
Test Loss=0.3347, Test acc=0.9114

Epoch: 3
[Step=50]	Loss=0.0615	acc=0.9798	5324.6 examples/second
[Step=100]	Loss=0.0608	acc=0.9797	5328.0 examples/second
[Step=150]	Loss=0.0603	acc=0.9798	5345.1 examples/second
Test Loss=0.3328, Test acc=0.9119

Epoch: 4
[Step=50]	Loss=0.0760	acc=0.9738	5325.8 examples/second
[Step=100]	Loss=0.0741	acc=0.9746	5300.4 examples/second
[Step=150]	Loss=0.072

In [16]:
# Check sparsity of the final model, make sure it's 85%
net.load_state_dict(torch.load("net_after_iterative_prune.pt", map_location=device))

for name,layer in net.named_modules():
    if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
        # Your code here: can copy from previous question
        np_weight = layer.weight.data.cpu().numpy()
        zeros = np_weight.size - np.count_nonzero(np_weight)
        total = np_weight.size
        print('Sparsity of '+name+': '+str(zeros/total))

test(net)

Sparsity of head_conv.0.conv: 0.8495370370370371
Sparsity of body_op.0.conv1.0.conv: 0.8498263888888888
Sparsity of body_op.0.conv2.0.conv: 0.8498263888888888
Sparsity of body_op.1.conv1.0.conv: 0.8498263888888888
Sparsity of body_op.1.conv2.0.conv: 0.8498263888888888
Sparsity of body_op.2.conv1.0.conv: 0.8498263888888888
Sparsity of body_op.2.conv2.0.conv: 0.8498263888888888
Sparsity of body_op.3.conv1.0.conv: 0.8498263888888888
Sparsity of body_op.3.conv2.0.conv: 0.8499348958333334
Sparsity of body_op.4.conv1.0.conv: 0.8499348958333334
Sparsity of body_op.4.conv2.0.conv: 0.8499348958333334
Sparsity of body_op.5.conv1.0.conv: 0.8499348958333334
Sparsity of body_op.5.conv2.0.conv: 0.8499348958333334
Sparsity of body_op.6.conv1.0.conv: 0.8499891493055556
Sparsity of body_op.6.conv2.0.conv: 0.8499891493055556
Sparsity of body_op.7.conv1.0.conv: 0.8499891493055556
Sparsity of body_op.7.conv2.0.conv: 0.8499891493055556
Sparsity of body_op.8.conv1.0.conv: 0.8499891493055556
Sparsity of body

### Lab2 (e) Global iterative pruning

In [17]:
def global_prune_by_percentage(net, q=70.0):
    """
    Pruning the weight paramters by threshold.
    :param q: pruning percentile. 'q' percent of the least
    significant weight paramters will be pruned.
    """
    # A list to gather all the weights
    flattened_weights = []
    # Find global pruning threshold
    for name,layer in net.named_modules():
        if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
            # Convert weight to numpy
            np_weight = layer.weight.data.cpu().numpy()

            # Flatten the weight and append to flattened_weights
            np_weight_flatten = np.abs(np_weight).flatten()
            flattened_weights.append(np_weight_flatten)

    # Concatenate all weights into a single numpy array
    flattened_weights = np.concatenate(flattened_weights)
    # Find global pruning threshold
    thres = np.percentile(flattened_weights, q)

    # Apply pruning threshold to all layers
    for name,layer in net.named_modules():
        if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
            # Convert weight to numpy
            weights = layer.weight.data.cpu().numpy()

            # Generate a binary mask same shape as weight to decide which element to prune
            mask = (np.abs(weights) > thres).astype(np.float32)

            # Convert mask to torch tensor and put on GPU
            mask_tensor = torch.from_numpy(mask).to(layer.weight.device)

            # Multiply the weight by mask to perform pruning
            layer.weight.data.mul_(mask_tensor)

In [18]:
net.load_state_dict(torch.load("pretrained_model.pt", map_location=device))
best_acc = 0.
for epoch in range(20):
    print('\nEpoch: %d' % epoch)

    net.train()
    if epoch<10:
        # Increase model sparsity
        q = 8.5 * (epoch + 1)
        global_prune_by_percentage(net, q=q)
    if epoch<9:
        finetune_after_prune(net, trainloader, criterion, optimizer,prune=False)
    else:
        finetune_after_prune(net, trainloader, criterion, optimizer)

    #Start the testing code.
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    num_val_steps = len(testloader)
    val_acc = correct / total
    print("Test Loss=%.4f, Test acc=%.4f" % (test_loss / (num_val_steps), val_acc))

    if epoch>=10:
        if val_acc > best_acc:
            best_acc = val_acc
            print("Saving...")
            torch.save(net.state_dict(), "net_after_global_iterative_prune.pt")


Epoch: 0
[Step=50]	Loss=0.0516	acc=0.9820	5025.7 examples/second
[Step=100]	Loss=0.0514	acc=0.9831	5057.1 examples/second
[Step=150]	Loss=0.0517	acc=0.9829	5055.1 examples/second
Test Loss=0.3247, Test acc=0.9144

Epoch: 1
[Step=50]	Loss=0.0521	acc=0.9834	4967.4 examples/second
[Step=100]	Loss=0.0503	acc=0.9842	4914.5 examples/second
[Step=150]	Loss=0.0491	acc=0.9845	4842.8 examples/second
Test Loss=0.3255, Test acc=0.9155

Epoch: 2
[Step=50]	Loss=0.0508	acc=0.9830	4745.6 examples/second
[Step=100]	Loss=0.0490	acc=0.9840	4693.3 examples/second
[Step=150]	Loss=0.0487	acc=0.9842	4403.4 examples/second
Test Loss=0.3266, Test acc=0.9131

Epoch: 3
[Step=50]	Loss=0.0563	acc=0.9827	4593.0 examples/second
[Step=100]	Loss=0.0573	acc=0.9819	4595.0 examples/second
[Step=150]	Loss=0.0566	acc=0.9826	4646.5 examples/second
Test Loss=0.3257, Test acc=0.9146

Epoch: 4
[Step=50]	Loss=0.0635	acc=0.9781	4833.7 examples/second
[Step=100]	Loss=0.0613	acc=0.9791	4871.5 examples/second
[Step=150]	Loss=0.061

In [19]:
net.load_state_dict(torch.load("net_after_global_iterative_prune.pt", map_location=device))

zeros_sum = 0
total_sum = 0
for name,layer in net.named_modules():
    if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
        # Your code here:
        # Convert the weight of "layer" to numpy array
        np_weight = layer.weight.data.cpu().numpy()
        # Count number of zeros
        zeros = np_weight.size - np.count_nonzero(np_weight)
        # Count number of parameters
        total = np_weight.size
        zeros_sum+=zeros
        total_sum+=total
        print('Sparsity of '+name+': '+str(zeros/total))
print('Total sparsity of: '+str(zeros_sum/total_sum))
test(net)

Sparsity of head_conv.0.conv: 0.3449074074074074
Sparsity of body_op.0.conv1.0.conv: 0.7092013888888888
Sparsity of body_op.0.conv2.0.conv: 0.6961805555555556
Sparsity of body_op.1.conv1.0.conv: 0.6805555555555556
Sparsity of body_op.1.conv2.0.conv: 0.7035590277777778
Sparsity of body_op.2.conv1.0.conv: 0.6796875
Sparsity of body_op.2.conv2.0.conv: 0.7369791666666666
Sparsity of body_op.3.conv1.0.conv: 0.689453125
Sparsity of body_op.3.conv2.0.conv: 0.7467447916666666
Sparsity of body_op.4.conv1.0.conv: 0.7842881944444444
Sparsity of body_op.4.conv2.0.conv: 0.8381076388888888
Sparsity of body_op.5.conv1.0.conv: 0.7858072916666666
Sparsity of body_op.5.conv2.0.conv: 0.8655598958333334
Sparsity of body_op.6.conv1.0.conv: 0.79931640625
Sparsity of body_op.6.conv2.0.conv: 0.823974609375
Sparsity of body_op.7.conv1.0.conv: 0.8343370225694444
Sparsity of body_op.7.conv2.0.conv: 0.8790961371527778
Sparsity of body_op.8.conv1.0.conv: 0.9013943142361112
Sparsity of body_op.8.conv2.0.conv: 0.987

In [20]:
net.load_state_dict(torch.load("net_after_global_iterative_prune.pt", map_location=device))

zeros_sum = 0
total_sum = 0
for name,layer in net.named_modules():
    if (isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear)) and 'id_mapping' not in name:
        # Your code here:
        # Convert the weight of "layer" to numpy array
        np_weight = layer.weight.data.cpu().numpy()
        # Count number of zeros
        zeros = np_weight.size - np.count_nonzero(np_weight)
        # Count number of parameters
        total = np_weight.size
        zeros_sum+=zeros
        total_sum+=total
        print('Sparsity of '+name+': '+str(zeros/total))
print('Total sparsity of: '+str(zeros_sum/total_sum))
test(net)

Sparsity of head_conv.0.conv: 0.3449074074074074
Sparsity of body_op.0.conv1.0.conv: 0.7092013888888888
Sparsity of body_op.0.conv2.0.conv: 0.6961805555555556
Sparsity of body_op.1.conv1.0.conv: 0.6805555555555556
Sparsity of body_op.1.conv2.0.conv: 0.7035590277777778
Sparsity of body_op.2.conv1.0.conv: 0.6796875
Sparsity of body_op.2.conv2.0.conv: 0.7369791666666666
Sparsity of body_op.3.conv1.0.conv: 0.689453125
Sparsity of body_op.3.conv2.0.conv: 0.7467447916666666
Sparsity of body_op.4.conv1.0.conv: 0.7842881944444444
Sparsity of body_op.4.conv2.0.conv: 0.8381076388888888
Sparsity of body_op.5.conv1.0.conv: 0.7858072916666666
Sparsity of body_op.5.conv2.0.conv: 0.8655598958333334
Sparsity of body_op.6.conv1.0.conv: 0.79931640625
Sparsity of body_op.6.conv2.0.conv: 0.823974609375
Sparsity of body_op.7.conv1.0.conv: 0.8343370225694444
Sparsity of body_op.7.conv2.0.conv: 0.8790961371527778
Sparsity of body_op.8.conv1.0.conv: 0.9013943142361112
Sparsity of body_op.8.conv2.0.conv: 0.987

### Lab 3 (b) and (c): Fixed-point quantization

In [ ]:
Nbits = 5 # Change this value to 5, 4, 3, 2 to finish Lab3 (b) and (c)
net = ResNetCIFAR(num_layers=20, Nbits=Nbits) # Define quantized model
net = net.to(device)
net.load_state_dict(torch.load("pretrained_model.pt", map_location=device))
test(net)


In [ ]:
# Quantized model finetuning
finetune(net, epochs=20, batch_size=256, lr=0.002, reg=1e-4)

# Load the model with best accuracy
net.load_state_dict(torch.load("quantized_net_after_finetune.pt", map_location=device))
test(net)

### Lab3 (d) Quantize pruned model
Load the 85% sparsity model from Lab2 (e).

In [21]:
Nbits = 5 # Change this value to 5, 4, 3, 2 to finish Lab3 (d)
net = ResNetCIFAR(num_layers=20, Nbits=Nbits) # Define quantized model
net = net.to(device)
net.load_state_dict(torch.load("net_after_global_iterative_prune.pt", map_location=device))
test(net)


TypeError: unsupported operand type(s) for *: 'ellipsis' and 'ellipsis'

In [ ]:
# Quantized model finetuning, keeping the pruned weights at zero
finetune(net, epochs=20, batch_size=256, lr=0.002, reg=1e-4, keep_sparsity=True)

# Load the model with best accuracy
net.load_state_dict(torch.load("quantized_net_after_finetune.pt", map_location=device))
test(net)